### Take Luhman (2022a, b) tables and determine the number of sources in Lupus and UpperSco w/ discs.

In [1]:
import re
import numpy  as np
import pandas as pd
pd.set_option('display.max_columns', None)

`Task #1:` Number of stars in Lupus and Upper Sco according to Luhman (kinematically and kinematically + astrometrically).

In [2]:
# Load Luhman22a's table:
luhman_stars = pd.read_fwf(
    "table1_luhman2022a_orig.dat",
    names  = [
        "SourceID_GaiaDR3", "SourceID_GaiaDR2", "NameOther",
        "RAdeg", "DEdeg",
        "SpType", "r_SpType", "AdoptSpType",
        "EWLi", "r_EWLi",
        "pmRA", "e_pmRA", "pmDec", "e_pmDec", "plx", "e_plx",
        "r_med_geo", "r_lo_geo", "r_hi_geo",
        "RVel", "e_RVel", "r_RVel",
        "U U", "e_U", "V V", "e_V", "W W", "e_W",
        "Gmag G", "e_Gmag", "GBPmag", "e_GBPmag", "GRPmag", "e_GRPmag",
        "RUWE", "kin", "pos",
        "2MASS", "SourceID_2MASS", "2MASS_angsep", "2MASS_closest",
        "WISE", "SourceID_WISE", "WISE_angsep", "WISE_closest"
    ],
    skiprows = 273,
)
print(len(luhman_stars))
# Numbers match: 10509, yes!

10509


In [3]:
# Kinematic members of Upper Sco:
mask_kin_usco = luhman_stars["kin"].str.contains(("u"))
print("Kinematic members of Upper Sco:", len(luhman_stars["kin"][mask_kin_usco].tolist()))
# Kinematic members of Lupus:
mask_kin_lupo = luhman_stars["kin"].str.contains(("l"))
print("Kinematic members of Lupus:", len(luhman_stars["kin"][mask_kin_lupo].tolist()))
# Shared members:
mask_kin_lu   = luhman_stars["kin"].str.contains(("ul"))
print("Kinematic members of both Upper Sco and Lupus:", len(luhman_stars["kin"][mask_kin_lu].tolist()))

Kinematic members of Upper Sco: 3103
Kinematic members of Lupus: 916
Kinematic members of both Upper Sco and Lupus: 795


In [4]:
# Kinematic and astrometric members of Upper Sco:
mask_kinpos_usco = (luhman_stars["kin"].str.contains(("u"))) & (luhman_stars["pos"].str.contains(("u")))
print("Kinematic and astrometric members of Upper Sco:", len(luhman_stars["kin"][mask_kinpos_usco].tolist()))
# Kinematic and astrometric members of Lupus:
mask_kinpos_lupo = (luhman_stars["kin"].str.contains(("l"))) & (luhman_stars["pos"].str.contains(("l")))
print("Kinematic and astrometric members of Lupus:", len(luhman_stars["kin"][mask_kinpos_lupo].tolist()))
# Shared members:
mask_kinpos_lu   = (luhman_stars["kin"].str.contains(("ul"))) & (luhman_stars["pos"].str.contains(("ul")))
print("Kinematic and astrometric members of both Upper Sco and Lupus:", len(luhman_stars["kin"][mask_kinpos_lu].tolist()))

Kinematic and astrometric members of Upper Sco: 1349
Kinematic and astrometric members of Lupus: 120
Kinematic and astrometric members of both Upper Sco and Lupus: 0


`Task #2:` Number of stars in Lupus and Upper Sco according to Luhman (kinematically and kinematically + astrometrically) with a counterpart in WISEA.

In [5]:
# Load Luhman22b's table:
luhman_discs = pd.read_fwf(
    "table1_luhman2022b_orig.dat",
    names  = [
        "WISE", "SourceID_WISE_discs", "RAdeg","DEdeg", "2MASS", "SourceID_2MASS_discs", "2MASS_angsep",
        "Jmag 2MASS", "e_Jmag", "Hmag 2MASS", "e_Hmag", "Ksmag 2MASS", "e_Ksmag",
        "W1mag", "e_W1mag", "W2mag", "e_W2mag", "W3mag", "e_W3mag", "f_W3mag", "W4mag", "e_W4mag", "f_W4mag",
        "ExcW2 Excess present in W2?", "ExcW3 Excess present in W3?", "ExcW4 Excess present in W4?", "DiskType Disk Typec"
    ],
    skiprows = 45,
)
print(len(luhman_discs))
# Numbers match: 9675, yes!

9675


In [6]:
# matched = luhman_discs.merge(
#     luhman_stars[["SourceID_2MASS", "2MASS_closest", "SourceID_WISE", "WISE_closest", "SourceID_GaiaDR3", "kin", "pos", "AdoptSpType"]],
#     on  = "SourceID_2MASS",
#     how = "left"
# )
# print(len(matched)) #17238
# This crazy number is most likely driven by the presence of NaNs in both columns. 
# Table1 of Luhman (2022b) is built based on WISE, thus, sources with 2MASS in Table1 of Luhman (2022a) but not WISE are not included.
# You have to match on WISE!

# matched = luhman_discs.merge(
#     luhman_stars[["SourceID_WISE", "WISE_closest", "SourceID_2MASS", "2MASS_closest", "SourceID_GaiaDR3", "kin", "pos", "AdoptSpType"]],
#     on  = "SourceID_WISE",
#     how = "left"
# )
# print(len(matched)) #10020
# This is also wrong because often only a single WISE ID - 2MASS ID combination is present in Table1 of Luhman (2022b) 
# while multiple are in Table1 of Luhman (2022a). Matching just by WISE will include both.
# There are 345 duplicated sources (i.e., more Gaia sources are associated to the same WISE ref ID).

In [7]:
# Remove those objects with no WISE counterpart in Luhman (2022a):
luhman_stars_match = luhman_stars.dropna(subset=["SourceID_WISE"])

# Match to add Gaia IDs, kin, pos info:
matched = luhman_discs.merge(
    luhman_stars_match[["SourceID_WISE", "WISE_closest", "SourceID_2MASS", "2MASS_closest", "SourceID_GaiaDR3", "kin", "pos", "AdoptSpType"]],
    left_on  = ["SourceID_WISE_discs", "SourceID_2MASS_discs"],
    right_on = ["SourceID_WISE", "SourceID_2MASS"],
    how      = "left"
)
print(len(matched)) #9970

# For some weird reason there are some WISE IDs in Table 1 of Luhman (2022b) with no counterpart in Luhman (2022a)...
# These are matched to nan nan and I'll drop them since have no association with a Gaia source/kinpos info.
# WISEA  J121959.36-501839.6  184.9973407 -50.3110062 2MASS J12195937-5018380
# is present in Luhman (2022a) as
# WISEA  J121959.36-501839.6  184.9973407 -50.3110062 2MASS J12195938-5018404
# and I updated it accordingly in Luhman (2022b)
num_dropped = matched[["SourceID_WISE","SourceID_2MASS"]].isna().all(axis=1).sum()
print("WISE IDs in Table 1 of Luhman (2022b) with no counterpart in Table 1 of Luhman (2022a):", num_dropped)

matched = matched.dropna(subset=["SourceID_WISE","SourceID_2MASS"],how="all")
print(len(matched)) #9955

# Duplicates?
dups = matched[matched["SourceID_WISE"].duplicated(keep=False)]
print(len(np.unique(dups["SourceID_WISE"].tolist())))
# There are 295 duplicated sources (i.e., more Gaia sources are associated to the same WISE ref ID).
# Those share the same WISE ID - 2MASS ID combination is present in Table1 of Luhman (2022b)

# Perfect 'cause 9970 - 295 is 9675, i.e., the size of Table 1 of Luhman (2022b).
# with open("matched.dat", "w") as f:
#     f.write(matched.to_string(index=False))

9970
WISE IDs in Table 1 of Luhman (2022b) with no counterpart in Table 1 of Luhman (2022a): 15
9955
295


In [8]:
# Those are genuine duplicates (same 2MASS and WISEA IDs, different GaiaEDR3 IDs) and include all kind of disc classifications.
np.unique(dups["SourceID_WISE"].tolist())
# with open("matched_dups.dat", "w") as f:
#     f.write(dups.to_string(index=False))

array(['J115151.15-651221.5', 'J115518.54-500906.3',
       'J115525.33-690441.0', 'J115834.02-435100.8',
       'J120036.25-492926.0', 'J120347.20-532406.6',
       'J120525.84-372620.8', 'J120721.91-533921.2',
       'J120858.15-513018.3', 'J121138.09-711036.1',
       'J121236.16-552000.8', 'J121756.73-490209.3',
       'J122031.21-650400.3', 'J122420.57-491208.3',
       'J122420.62-544354.1', 'J122433.33-631155.3',
       'J122638.45-592851.7', 'J122707.33-505220.9',
       'J122900.62-631218.8', 'J123023.17-503050.9',
       'J123056.27-655024.6', 'J123108.47-545644.3',
       'J123446.11-701925.8', 'J123452.38-565249.3',
       'J123911.57-590135.3', 'J123918.16-495951.6',
       'J123926.17-552718.8', 'J124017.03-494556.2',
       'J124034.70-684224.6', 'J124200.46-575948.7',
       'J124211.01-593543.3', 'J124356.50-533423.3',
       'J124559.08-680847.8', 'J124747.34-484303.1',
       'J124831.05-600227.2', 'J124848.13-563537.9',
       'J125016.84-513038.2', 'J125044.87-5654

`Task #3:` Split them into disc-bearing and disc-less.

In [9]:
# Discless stars:
excluded_class = ["debris/ev trans", "debris/ev trans?", "debris/ev-trans", "debris/ev-trans?", "debris", "III", "III?", "Be", "Be?"]

In [10]:
# The following Lupus targets are debris/evolved tr in Luhman (2022b) but have detected continuum/CO and resolved gas with ALMA. 
# Will update them to count as full discs (just to be properly considered in the counting)
# 2MASS J15580252-3736026 WISEA J155802.50-373602.9
# 2MASS J16071159-3903475 WISEA J160711.59-390347.7
# 2MASS J16085468-3937431 WISEA J160854.67-393743.3

print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J15580252-3736026')&(matched["SourceID_WISE"]=='J155802.50-373602.9')])
print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16071159-3903475')&(matched["SourceID_WISE"]=='J160711.59-390347.7')])
print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16085468-3937431')&(matched["SourceID_WISE"]=='J160854.67-393743.3')])

mask = (
    ((matched["SourceID_2MASS"] == "J15580252-3736026") & (matched["SourceID_WISE"] == "J155802.50-373602.9")) |
    ((matched["SourceID_2MASS"] == "J16071159-3903475") & (matched["SourceID_WISE"] == "J160711.59-390347.7")) |
    ((matched["SourceID_2MASS"] == "J16085468-3937431") & (matched["SourceID_WISE"] == "J160854.67-393743.3"))
)
# Inspect the matched rows:
# print(matched.loc[mask])     
matched.loc[mask,"DiskType Disk Typec"] = "full_riclassified"

print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J15580252-3736026')&(matched["SourceID_WISE"]=='J155802.50-373602.9')])
print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16071159-3903475')&(matched["SourceID_WISE"]=='J160711.59-390347.7')])
print(matched["DiskType Disk Typec"][(matched["SourceID_2MASS"]=='J16085468-3937431')&(matched["SourceID_WISE"]=='J160854.67-393743.3')])

5881    debris/ev-trans
Name: DiskType Disk Typec, dtype: object
6579    debris/ev-trans
Name: DiskType Disk Typec, dtype: object
6762    debris/ev-trans
Name: DiskType Disk Typec, dtype: object
5881    full_riclassified
Name: DiskType Disk Typec, dtype: object
6579    full_riclassified
Name: DiskType Disk Typec, dtype: object
6762    full_riclassified
Name: DiskType Disk Typec, dtype: object


In [11]:
# UpperSco kinematics
matched_usco_k  = matched[ matched["kin"].astype(str).str.contains("u", na=False) ]
print(len(matched_usco_k))
# UpperSco kinematics and astrometry
matched_usco_kp = matched[(matched["kin"].astype(str).str.contains("u", na=False)) & (matched["pos"].astype(str).str.contains("u", na=False))]
print(len(matched_usco_kp))

# Lupus kinematics
matched_lupo_k  = matched[ matched["kin"].astype(str).str.contains("l", na=False) ]
print(len(matched_lupo_k))
# Lupus kinematics and astrometry
matched_lupo_kp = matched[(matched["kin"].astype(str).str.contains("l", na=False)) & (matched["pos"].astype(str).str.contains("l", na=False))]
print(len(matched_lupo_kp))

2962
1323
860
119


In [12]:
# The membership of 13 Upper Sco targets was re-evaluated by Carpenter et al. (2025) based on their ALMA data.
# The following are likely to be background objects and will be removed:
excludeobjs2MASS = ["J15442550-2126408", "J15564244-2039339", "J15594231-2945495", "J16044876-1748393", "J16050647-1734020", "J16153220-2010236", "J16185382-2053182", "J15524851-2845369", "J15594460-2155250", "J16052556-2035397", "J16064385-1908056", "J16120505-2043404", "J16185277-2259537"]

pK = matched_usco_k[matched_usco_k["SourceID_2MASS"].isin(excludeobjs2MASS)]
print(pK["SourceID_2MASS"].tolist(), pK["SourceID_WISE"].tolist())
pKP = matched_usco_kp[matched_usco_kp["SourceID_2MASS"].isin(excludeobjs2MASS)]
print(pKP["SourceID_2MASS"].tolist(), pKP["SourceID_WISE"].tolist())

mask            = matched_usco_k["SourceID_2MASS"].isin(excludeobjs2MASS)
matched_usco_k  = matched_usco_k[~mask].copy()
print(len(matched_usco_k["SourceID_2MASS"]))

mask            = matched_usco_kp["SourceID_2MASS"].isin(excludeobjs2MASS)
matched_usco_kp = matched_usco_kp[~mask].copy()
print(len(matched_usco_kp["SourceID_2MASS"]))

['J15564244-2039339', 'J15564244-2039339', 'J16044876-1748393', 'J16044876-1748393', 'J16050647-1734020', 'J16052556-2035397', 'J16064385-1908056', 'J16185277-2259537'] ['J155642.45-203934.1', 'J155642.45-203934.1', 'J160448.71-174838.9', 'J160448.71-174838.9', 'J160506.45-173402.5', 'J160525.55-203539.9', 'J160643.85-190805.8', 'J161852.77-225954.0']
['J15564244-2039339', 'J15564244-2039339', 'J16052556-2035397', 'J16064385-1908056', 'J16185277-2259537'] ['J155642.45-203934.1', 'J155642.45-203934.1', 'J160525.55-203539.9', 'J160643.85-190805.8', 'J161852.77-225954.0']
2954
1318


In [13]:
# Single targets:
unique_usco_k = matched_usco_k[~matched_usco_k["SourceID_WISE_discs"].duplicated(keep=False)]
# Duplicate targets:
dups_usco_k   = matched_usco_k[ matched_usco_k["SourceID_WISE_discs"].duplicated(keep=False)]
print(f"Check that length unique+duplicates matches table: {len(unique_usco_k)+len(dups_usco_k)}")
print(f"Length duplicates: {len(np.unique(dups_usco_k["SourceID_WISE_discs"]))}")

# Duplicate disc less: keep both stars (none has a disc)
dups_usco_k_discless    = dups_usco_k[ dups_usco_k["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc less   : {len(np.unique(dups_usco_k_discless["SourceID_WISE_discs"]))}")
# Duplicate disc bearing: consider both extreme scenarios - both have discs or closest match only)
dups_usco_k_discbearing = dups_usco_k[~dups_usco_k["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc bearing: {len(np.unique(dups_usco_k_discbearing["SourceID_WISE_discs"]))}")
print(
    f"Length duplicates closest disc bearing: {len(dups_usco_k_discbearing[dups_usco_k_discbearing["WISE_closest"]=="Y"])}"
)

# Make a table with the assumption that all the disc-bearing duplicates have both components disc-bearing
usco_k_both = pd.concat([unique_usco_k, dups_usco_k_discless, dups_usco_k_discbearing], ignore_index=True)
print(f"Number of stars both  : {len(usco_k_both)}")
# Make a table with the assumption that all the disc-bearing duplicates have one component disc-bearing (the closest)
usco_k_sing = pd.concat([unique_usco_k, dups_usco_k_discless, dups_usco_k_discbearing[dups_usco_k_discbearing["WISE_closest"]=="Y"]], ignore_index=True)
print(f"Number of stars single: {len(usco_k_sing)}")

# Select disc-bearing stars in each catalogue
print(f"Number of stars w/ discs both  : {len(usco_k_both[~usco_k_both["DiskType Disk Typec"].isin(excluded_class)])}")
print(f"Number of stars w/ discs single: {len(usco_k_sing[~usco_k_sing["DiskType Disk Typec"].isin(excluded_class)])}")

Check that length unique+duplicates matches table: 2954
Length duplicates: 74
Length duplicates disc less   : 61
Length duplicates disc bearing: 13
Length duplicates closest disc bearing: 13
Number of stars both  : 2954
Number of stars single: 2941
Number of stars w/ discs both  : 547
Number of stars w/ discs single: 534


In [14]:
# Check that you only included disc-bearing sources:
usco_k_both[~usco_k_both["DiskType Disk Typec"].isin(excluded_class)]["DiskType Disk Typec"].tolist()

['full',
 'full',
 'transitional',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full',
 'full',
 'ev-or-trans',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full/edge-on?',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full/edge-on?',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full',
 'full',
 'full',
 'full/edge-on?',
 'transitional',
 'full',
 'full',
 'full',
 'full',
 'full',
 'transitional',
 'full_riclassified',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full'

In [15]:
# Single targets:
unique_usco_kp = matched_usco_kp[~matched_usco_kp["SourceID_WISE_discs"].duplicated(keep=False)]
# Duplicate targets:
dups_usco_kp   = matched_usco_kp[ matched_usco_kp["SourceID_WISE_discs"].duplicated(keep=False)]
print(f"Check that length unique+duplicates matches table: {len(unique_usco_kp)+len(dups_usco_kp)}")
print(f"Length duplicates: {len(np.unique(dups_usco_kp["SourceID_WISE_discs"]))}")

# Duplicate disc less: keep both stars (none has a disc)
dups_usco_kp_discless    = dups_usco_kp[ dups_usco_kp["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc less   : {len(np.unique(dups_usco_kp_discless["SourceID_WISE_discs"]))}")
# Duplicate disc bearing: consider both extreme scenarios - both have discs or closest match only)
dups_usco_kp_discbearing = dups_usco_kp[~dups_usco_kp["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc bearing: {len(np.unique(dups_usco_kp_discbearing["SourceID_WISE_discs"]))}")
print(
    f"Length duplicates closest disc bearing: {len(dups_usco_kp_discbearing[dups_usco_kp_discbearing["WISE_closest"]=="Y"])}"
)

# Make a table with the assumption that all the disc-bearing duplicates have both components disc-bearing
usco_kp_both = pd.concat([unique_usco_kp, dups_usco_kp_discless, dups_usco_kp_discbearing], ignore_index=True)
print(f"Number of stars both  : {len(usco_kp_both)}")
# Make a table with the assumption that all the disc-bearing duplicates have one component disc-bearing (the closest)
usco_kp_sing = pd.concat([unique_usco_kp, dups_usco_kp_discless, dups_usco_kp_discbearing[dups_usco_kp_discbearing["WISE_closest"]=="Y"]], ignore_index=True)
print(f"Number of stars single: {len(usco_kp_sing)}")

# Select disc-bearing stars in each catalogue
print(f"Number of stars w/ discs both  : {len(usco_kp_both[~usco_kp_both["DiskType Disk Typec"].isin(excluded_class)])}")
print(f"Number of stars w/ discs single: {len(usco_kp_sing[~usco_kp_sing["DiskType Disk Typec"].isin(excluded_class)])}")

Check that length unique+duplicates matches table: 1318
Length duplicates: 29
Length duplicates disc less   : 24
Length duplicates disc bearing: 5
Length duplicates closest disc bearing: 5
Number of stars both  : 1318
Number of stars single: 1313
Number of stars w/ discs both  : 257
Number of stars w/ discs single: 252


In [16]:
# Check that you only included disc-bearing sources:
usco_kp_both[~usco_kp_both["DiskType Disk Typec"].isin(excluded_class)]["DiskType Disk Typec"].tolist()

['full',
 'full/edge-on?',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full/edge-on?',
 'full',
 'full',
 'full',
 'full',
 'full',
 'transitional',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'transitional',
 'ev-or-trans',
 'evolved',
 'full',
 'full',
 'transitional',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'transitional',
 'full',
 'full',
 'full/edge-on?',
 'full/edge-on?',
 'full',
 'full',
 'transitional',
 'full',
 'full',
 'full',
 'full',
 'full/edge-on?',
 'full',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 

In [17]:
# Single targets:
unique_lupo_k = matched_lupo_k[~matched_lupo_k["SourceID_WISE_discs"].duplicated(keep=False)]
# Duplicate targets:
dups_lupo_k   = matched_lupo_k[ matched_lupo_k["SourceID_WISE_discs"].duplicated(keep=False)]
print(f"Check that length unique+duplicates matches table: {len(unique_lupo_k)+len(dups_lupo_k)}")
print(f"Length duplicates: {len(np.unique(dups_lupo_k["SourceID_WISE_discs"]))}")

# Duplicate disc less: keep both stars (none has a disc)
dups_lupo_k_discless    = dups_lupo_k[ dups_lupo_k["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc less   : {len(np.unique(dups_lupo_k_discless["SourceID_WISE_discs"]))}")
# Duplicate disc bearing: consider both extreme scenarios - both have discs or closest match only)
dups_lupo_k_discbearing = dups_lupo_k[~dups_lupo_k["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc bearing: {len(np.unique(dups_lupo_k_discbearing["SourceID_WISE_discs"]))}")
print(
    f"Length duplicates closest disc bearing: {len(dups_lupo_k_discbearing[dups_lupo_k_discbearing["WISE_closest"]=="Y"])}"
)

# Make a table with the assumption that all the disc-bearing duplicates have both components disc-bearing
lupo_k_both = pd.concat([unique_lupo_k, dups_lupo_k_discless, dups_lupo_k_discbearing], ignore_index=True)
print(f"Number of stars both  : {len(lupo_k_both)}")
# Make a table with the assumption that all the disc-bearing duplicates have one component disc-bearing (the closest)
lupo_k_sing = pd.concat([unique_lupo_k, dups_lupo_k_discless, dups_lupo_k_discbearing[dups_lupo_k_discbearing["WISE_closest"]=="Y"]], ignore_index=True)
print(f"Number of stars single: {len(lupo_k_sing)}")

# Select disc-bearing stars in each catalogue
print(f"Number of stars w/ discs both  : {len(lupo_k_both[~lupo_k_both["DiskType Disk Typec"].isin(excluded_class)])}")
print(f"Number of stars w/ discs single: {len(lupo_k_sing[~lupo_k_sing["DiskType Disk Typec"].isin(excluded_class)])}")

Check that length unique+duplicates matches table: 860
Length duplicates: 16
Length duplicates disc less   : 11
Length duplicates disc bearing: 5
Length duplicates closest disc bearing: 5
Number of stars both  : 860
Number of stars single: 855
Number of stars w/ discs both  : 185
Number of stars w/ discs single: 180


In [18]:
# Check that you only included disc-bearing sources:
lupo_k_both[~lupo_k_both["DiskType Disk Typec"].isin(excluded_class)]["DiskType Disk Typec"].tolist()

['full',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full',
 'transitional',
 'full',
 'full',
 'full',
 'full_riclassified',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'evolved',
 'transitional',
 'transitional',
 'full/edge-on?',
 'transitional',
 'full',
 'full_riclassified',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'ev-or-trans',
 'full',
 'ev-or-trans',
 'full',
 'full',
 '

In [19]:
# Single targets:
unique_lupo_kp = matched_lupo_kp[~matched_lupo_kp["SourceID_WISE_discs"].duplicated(keep=False)]
# Duplicate targets:
dups_lupo_kp   = matched_lupo_kp[ matched_lupo_kp["SourceID_WISE_discs"].duplicated(keep=False)]
print(f"Check that length unique+duplicates matches table: {len(unique_lupo_kp)+len(dups_lupo_kp)}")
print(f"Length duplicates: {len(np.unique(dups_lupo_kp["SourceID_WISE_discs"]))}")

# Duplicate disc less: keep both stars (none has a disc)
dups_lupo_kp_discless    = dups_lupo_kp[ dups_lupo_kp["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc less   : {len(np.unique(dups_lupo_kp_discless["SourceID_WISE_discs"]))}")
# Duplicate disc bearing: consider both extreme scenarios - both have discs or closest match only)
dups_lupo_kp_discbearing = dups_lupo_kp[~dups_lupo_kp["DiskType Disk Typec"].isin(excluded_class)]
print(f"Length duplicates disc bearing: {len(np.unique(dups_lupo_kp_discbearing["SourceID_WISE_discs"]))}")
print(
    f"Length duplicates closest disc bearing: {len(dups_lupo_kp_discbearing[dups_lupo_kp_discbearing["WISE_closest"]=="Y"])}"
)

# Make a table with the assumption that all the disc-bearing duplicates have both components disc-bearing
lupo_kp_both = pd.concat([unique_lupo_kp, dups_lupo_kp_discless, dups_lupo_kp_discbearing], ignore_index=True)
print(f"Number of stars both  : {len(lupo_kp_both)}")
# Make a table with the assumption that all the disc-bearing duplicates have one component disc-bearing (the closest)
lupo_kp_sing = pd.concat([unique_lupo_kp, dups_lupo_kp_discless, dups_lupo_kp_discbearing[dups_lupo_kp_discbearing["WISE_closest"]=="Y"]], ignore_index=True)
print(f"Number of stars single: {len(lupo_kp_sing)}")

# Select disc-bearing stars in each catalogue
print(f"Number of stars w/ discs both  : {len(lupo_kp_both[~lupo_kp_both["DiskType Disk Typec"].isin(excluded_class)])}")
print(f"Number of stars w/ discs single: {len(lupo_kp_sing[~lupo_kp_sing["DiskType Disk Typec"].isin(excluded_class)])}")

Check that length unique+duplicates matches table: 119
Length duplicates: 5
Length duplicates disc less   : 2
Length duplicates disc bearing: 3
Length duplicates closest disc bearing: 3
Number of stars both  : 119
Number of stars single: 116
Number of stars w/ discs both  : 76
Number of stars w/ discs single: 73


In [20]:
# Check that you only included disc-bearing sources:
lupo_kp_both[~lupo_kp_both["DiskType Disk Typec"].isin(excluded_class)]["DiskType Disk Typec"].tolist()

['full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'transitional',
 'full',
 'full',
 'full_riclassified',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full_riclassified',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'evolved',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full_riclassified',
 'full',
 'transitional',
 'full',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'full',
 'full',
 'full',
 'evolved',
 'full',
 'transitional',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full',
 'full']

`Task 4:` Number of k and kp member in the catalogue of Carpenter+25 with a COflux or a COsize.

In [21]:
# Load your extended version of the table:
uppersco_mine = pd.read_csv(
    "uppersco_vs_polnitzky.dat",
    names  = [
        "Region", "Name", "2MASS", "SourceID_2MASS", "WISE", "SourceID_WISE", "GaiaDR3", "SourceID_GaiaDR3",
        "SpT_E26", "SpT_L22", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22", 
        "kinpopL22", "pospopL22", "memK_L22", "memKP_L22", "discClassC25", "discClassL22", "COsizeS21?", "comments"
    ],
    sep      = r"\s+",
    skiprows = 1
)
print(len(uppersco_mine))
# Numbers match: 284.

284


In [22]:
uppersco_mine_DB = uppersco_mine[~uppersco_mine["discClassL22"].isin(excluded_class)]

In [23]:
print(f"Stars with discs targeted in UpperSco K :", len(uppersco_mine_DB[uppersco_mine_DB["memK_L22" ]=='Y']))
print(f"Stars with discs targeted in UpperSco KP:", len(uppersco_mine_DB[uppersco_mine_DB["memKP_L22"]=='Y']))
print(f"Stars with discs resolved in UpperSco K :", len(uppersco_mine_DB[(uppersco_mine_DB["memK_L22" ]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y')]))
print(f"Stars with discs resolved in UpperSco KP:", len(uppersco_mine_DB[(uppersco_mine_DB["memKP_L22"]=='Y') & (uppersco_mine_DB["COsizeS21?"]=='Y')]))

Stars with discs targeted in UpperSco K : 164
Stars with discs targeted in UpperSco KP: 140
Stars with discs resolved in UpperSco K : 69
Stars with discs resolved in UpperSco KP: 62


In [24]:
# Load your extended version of the table:
lupus_mine = pd.read_csv(
    "lupus_vs_polnitzky.dat",
    skiprows = 49,
    sep      = r'\s+',
    names   = [
        "Region", "Name", "2MASS", "SourceID_2MASS", "WISE", "SourceID_WISE", "GaiaDR3", "SourceID_GaiaDR3",
        "SpT_E26", "SpT_L22", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22", 
        "kinpopL22", "pospopL22", "memK_L22", "memKP_L22", "mem_T22", "discClassL22", "COsizeS21?", "comments"
    ]
)
print(len(lupus_mine))
# Numbers match: 100, yes!

98


In [25]:
# The following Lupus targets are debris/evolved tr in Luhman (2022b) but have detected continuum/CO and resolved gas with ALMA. 
# Will update them to count as full discs (just to be properly considered in the counting)
# 2MASS J15580252-3736026 WISEA J155802.50-373602.9
# 2MASS J16071159-3903475 WISEA J160711.59-390347.7
# 2MASS J16085468-3937431 WISEA J160854.67-393743.3

print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J15580252-3736026')&(lupus_mine["SourceID_WISE"]=='J155802.50-373602.9')])
print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16071159-3903475')&(lupus_mine["SourceID_WISE"]=='J160711.59-390347.7')])
print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16085468-3937431')&(lupus_mine["SourceID_WISE"]=='J160854.67-393743.3')])

mask = (
    ((lupus_mine["SourceID_2MASS"] == "J15580252-3736026") & (lupus_mine["SourceID_WISE"] == "J155802.50-373602.9")) |
    ((lupus_mine["SourceID_2MASS"] == "J16071159-3903475") & (lupus_mine["SourceID_WISE"] == "J160711.59-390347.7")) |
    ((lupus_mine["SourceID_2MASS"] == "J16085468-3937431") & (lupus_mine["SourceID_WISE"] == "J160854.67-393743.3"))
)
# Inspect the matched rows:
# print(matched.loc[mask])     
lupus_mine.loc[mask,"discClassL22"] = "full_riclassified"

print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J15580252-3736026')&(lupus_mine["SourceID_WISE"]=='J155802.50-373602.9')])
print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16071159-3903475')&(lupus_mine["SourceID_WISE"]=='J160711.59-390347.7')])
print(lupus_mine["discClassL22"][(lupus_mine["SourceID_2MASS"]=='J16085468-3937431')&(lupus_mine["SourceID_WISE"]=='J160854.67-393743.3')])

20    debris/ev-trans
Name: discClassL22, dtype: object
37    debris/ev-trans
Name: discClassL22, dtype: object
64    debris/ev-trans
Name: discClassL22, dtype: object
20    full_riclassified
Name: discClassL22, dtype: object
37    full_riclassified
Name: discClassL22, dtype: object
64    full_riclassified
Name: discClassL22, dtype: object


In [26]:
lupus_mine_DB = lupus_mine[~lupus_mine["discClassL22"].isin(excluded_class)]

In [27]:
print(f"Stars with discs targeted in Lupus K :", len(lupus_mine_DB[lupus_mine_DB["memK_L22" ]=='Y']))
print(f"Stars with discs targeted in Lupus KP:", len(lupus_mine_DB[lupus_mine_DB["memKP_L22"]=='Y']))
print(f"Stars with discs resolved in Lupus K :", len(lupus_mine_DB[(lupus_mine_DB["memK_L22" ]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y')]))
print(f"Stars with discs resolved in Lupus KP:", len(lupus_mine_DB[(lupus_mine_DB["memKP_L22"]=='Y') & (lupus_mine_DB["COsizeS21?"]=='Y')]))

Stars with discs targeted in Lupus K : 62
Stars with discs targeted in Lupus KP: 59
Stars with discs resolved in Lupus K : 44
Stars with discs resolved in Lupus KP: 41
